# Statistical Analysis
This notebook handles formal statistical tests to understand relationships between attack features.

### Load Cleaned Data
Loading data from `data/processed/gtd_cleaned.csv`.

In [ ]:
import pandas as pd
import scipy.stats as stats
import os
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(''))) if '__file__' not in globals() else os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
input_path = os.path.join(BASE_DIR, 'data', 'processed', 'gtd_cleaned.csv')
outputs_dir = os.path.join(BASE_DIR, 'notebooks', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

df = pd.read_csv(input_path)
print(f"Data shape: {df.shape}")
print(df.head().to_string())

## 1. Chi-Square Test: Region vs Attack Type
**Hypothesis Test:** Is there a statistically significant relationship between the region of an incident and its attack type?
- $H_0$: Region and attack type are independent.
- $H_a$: Region and attack type are dependent.

In [ ]:
contingency_table = pd.crosstab(df['region'], df['attack_type'])
chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
print(f"Chi2 Statistic: {chi2:.2f}")
print(f"P-value: {p:.4e}")

**Conclusion:** We reject the null hypothesis at 95% confidence (p < 0.05), meaning region significantly influences attack type. Different regions experience distinct patterns in types of attacks.

## 2. ANOVA Test: Mean Casualties by Attack Type
**Hypothesis Test:** Do mean casualties differ significantly across attack types?
- $H_0$: Mean casualties are the same across all attack types.
- $H_a$: At least one attack type has a different mean casualty rate.

In [ ]:
casualty_data = df.dropna(subset=['casualties', 'attack_type'])
groups = [group['casualties'].values for name, group in casualty_data.groupby('attack_type')]
f_stat, p_val = stats.f_oneway(*groups)
print(f"F-statistic: {f_stat:.2f}")
print(f"P-value: {p_val:.4e}")

**Conclusion:** We reject the null hypothesis at 95% confidence (p < 0.05), indicating a statistically significant difference in casualties between different attack types.

## 3. Pearson Correlation: Incidents vs Casualties
**Hypothesis Test:** What is the correlation between the number of incidents per year and total casualties per year?
- $H_0$: The correlation is 0.
- $H_a$: The correlation is non-zero.

In [ ]:
yearly_stats = df.groupby('year').agg(
    incidents=('event_id', 'count'),
    casualties=('casualties', 'sum')
).reset_index()

r_val, p_val_corr = stats.pearsonr(yearly_stats['incidents'], yearly_stats['casualties'])
print(f"Pearson r: {r_val:.4f}")
print(f"P-value: {p_val_corr:.4e}")

plt.figure(figsize=(10,6))
sns.scatterplot(data=yearly_stats, x='incidents', y='casualties')
plt.title('Incidents vs Casualties per Year')
plt.xlabel('Incidents')
plt.ylabel('Casualties')
plt.grid(True)
plt.savefig(os.path.join(outputs_dir, 'incidents_vs_casualties_scatter.png'))
plt.show()

**Conclusion:** The p-value is extremely low, and the r-value is highly positive, rejecting the null hypothesis. There is a strong, positive correlation between the frequency of incidents and the total casualties per year.

## 4. Year-over-Year (YoY) Trend Table
Calculating the percentage change in incidents from one year to the next. The result will be saved to `data/processed/yoy_trend.csv`.

In [ ]:
yearly_stats['yoy_change_pct'] = yearly_stats['incidents'].pct_change() * 100
output_yoy = os.path.join(BASE_DIR, 'data', 'processed', 'yoy_trend.csv')
os.makedirs(os.path.dirname(output_yoy), exist_ok=True)
yearly_stats.to_csv(output_yoy, index=False)

print("YoY Trend Table Output:")
print(yearly_stats.head().to_string())